# ARCHIVIST — the full pipeline, stage by stage

**auto trend discovery → reference mining → BFL Context → apparel graphics**

The bot chooses its own subject. Section 1 below is the discovery cycle that decides
*what is worth printing this week*; everything after it is the design work on the winner.

This notebook runs the same code the Gradio app runs, one stage at a time, so you can see
(and change) what happens between a topic and a print-ready shirt graphic.

| stage | module | what it decides |
|---|---|---|
| 0 | `discovery` | **which topic to design at all** — growth, social heat, competition |
| 1 | `trends` | the micro-niche the design occupies |
| 2 | `queries` | 10–30 varied searches across six clusters |
| 3–4 | `mining`, `analysis` | candidate references and their measured attributes |
| 5–6 | `scoring`, `roles` | which references survive, and what each one is *for* |
| 7 | `dna` | the Visual DNA profile |
| 8–9 | `direction` | the named art direction and the collection style lock |
| 10–11 | `variations`, `prompts`, `gate` | three ranked directions, prompts, quality gate |
| 12–13 | `bfl`, `apparel` | artwork, then the print package |
| 14 | `board`, `brief` | reference board, creative brief, run report |

**No API keys?** Set `OFFLINE = True` in the setup cell. Every stage still runs, using
procedurally generated stand-in references — the fastest way to understand the system
before spending credits.

## 0 · Environment

In [ ]:
# Run this once. In Colab or a fresh kernel it installs what is missing.
import importlib.util, subprocess, sys

REQUIRED = {"PIL": "Pillow", "requests": "requests", "gradio": "gradio"}
missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("installing:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# Make the package importable when the notebook is opened from notebooks/.
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "archivist").is_dir() and (ROOT.parent / "archivist").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("project root:", ROOT)

In [ ]:
import os

# --- the only settings you normally touch --------------------------------
# Leave TOPIC empty and section 1 discovers one; set it to override discovery.
TOPIC          = ""
AUDIENCE       = "design-literate streetwear buyers who read the label"
COLLECTION     = "notebook"          # collections share a style lock
GARMENT        = "dark"              # dark | faded-black | black | light | white | sand
AGGRESSIVENESS = 6                   # 0 restrained … 10 hostile
OFFLINE        = True                # True = synthetic references, no network, no keys

# Keys: set them in .env, or export them, or paste them here for this session only.
# os.environ["BFL_API_KEY"] = "..."
# os.environ["PEXELS_API_KEY"] = "..."
# os.environ["OPENAI_API_KEY"] = "..."   # GPT-5.1 creative assist

os.environ.setdefault("ARCHIVIST_RUNS_DIR", str(ROOT / "runs"))

from archivist.config import Settings

settings = Settings.from_env(ROOT, collection=COLLECTION, offline=OFFLINE, garment=GARMENT)
settings.capability_report()

### Connection self-test\n\nSame probes as the app's Connection tab. Optional services report yellow and degrade one stage; they never stop a run.

In [ ]:
from archivist import connectivity

results = connectivity.run_checks(settings)
for check in results:
    print(f"{check.badge:<10} {check.name:<28} {check.ms:>5} ms  {check.detail[:90]}")
print()
print(connectivity.summarise(results))

## 1 · Discovery — the bot picks the topic

Google Trends for three-month growth, Reddit / X / Meta for whether anyone cares, DuckDuckGo
for how crowded the apparel market already is. Everything that must not be printed — brands,
likenesses, tragedy, live politics — is screened out *before* it is measured.

In [ ]:
from archivist.discovery import DiscoveryEngine
from archivist.llm import LLM

llm = LLM(settings.openai_api_key, model=settings.openai_model,
          base_url=settings.openai_base_url, enabled=settings.can_use_llm)
engine = DiscoveryEngine(settings, llm=llm, log=print)
report = engine.discover(count=6)

print()
for opportunity in report.opportunities:
    print(f"{opportunity.overall:5.2f}  {opportunity.topic[:38]:<38} "
          f"growth {opportunity.growth_3m:+7.0f}%  social {opportunity.social_heat:5.1f}  "
          f"eng/post {opportunity.avg_engagement:7.0f}  competition {opportunity.competition:5.1f}")

In [ ]:
# Why the top pick won — every number carries the source that produced it.
best = report.best
if best:
    print(best.topic, "\n")
    for signal in best.signals:
        state = "ok " if signal.ok else "n/a"
        print(f"  [{state}] {signal.source:<14} {signal.metric:<12} {signal.detail}")
    print("\n  family:", best.family, "| moves with:", best.correlated_with)
    if best.angle:
        print("  angle:", best.angle)

In [ ]:
# What was thrown away, and why. An autonomous bot has to be auditable.
for rejected in report.rejected[:12]:
    print(f"  {rejected.topic[:44]:<44} {rejected.rejected_reason}")
print("\nfamilies:", report.families)

In [ ]:
# The rest of the notebook designs this topic.
TOPIC = TOPIC or (report.best.topic if report.best else "municipal water infrastructure")
AUDIENCE = (report.best.audience if report.best and report.best.audience else AUDIENCE)
print("designing:", TOPIC)

## 2 · Trend → micro-niche

A literal trend is a crowded market. The ladder escalates it until it names a place almost
nobody else occupies — and implies an institution that could plausibly have filed it.

In [ ]:
from archivist.trends import derive_ladder

ladder = derive_ladder(TOPIC, audience=AUDIENCE, aggressiveness=AGGRESSIVENESS, seed=settings.seed)

for rung in ("generic", "better", "niche", "micro_niche"):
    print(f"{rung:>12}: {getattr(ladder, rung)}")
print()
for signal in ladder.cultural_signals:
    print(" ·", signal)

## 3 · Search queries

Six clusters — literal subject, historical/archival, visual language, texture, typography,
composition. Near-duplicates are dropped by token overlap, which is what stops the classic
*forensic image / forensic photos / forensic photography* failure.

In [ ]:
from archivist.queries import cluster_summary, generate_queries

queries = generate_queries(TOPIC, ladder, count=settings.max_queries, seed=settings.seed)
for query in queries:
    print(f"[{query.cluster.value:<22}] {query.text}")
print()
print(cluster_summary(queries))

## 4 · Mining and measurement

Every downloaded candidate is measured with Pillow: contrast, grain, edge density, tonal
split, colourfulness, structure, type-likeness, palette, perceptual hash. Those numbers —
not adjectives — drive scoring, role assignment and the Visual DNA.

In [ ]:
from archivist.mining import mine
from archivist.pipeline import make_run_dir
from archivist.sources import build_sources

run_dir, slug = make_run_dir(settings, TOPIC)
sources = build_sources(settings)
print("sources:", [source.name for source in sources])

candidates, warnings = mine(
    queries, sources, run_dir / "references",
    per_query=settings.candidates_per_query,
    max_candidates=settings.max_candidates,
    request_delay=settings.request_delay,
    progress=lambda fraction, message: print(f"  {fraction:5.0%}  {message}", end="\r"),
)
print(f"\n{len(candidates)} candidates → {run_dir}")
for warning in warnings[:5]:
    print(" !", warning)

In [ ]:
from archivist.analysis import describe

for reference in candidates[:8]:
    print(f"{reference.source:<10} {describe(reference.attributes):<58} {reference.query[:40]}")

## 5 · Scoring and roles

Six weighted axes decide what survives (subject .20, distinctiveness .20, composition .15,
style .20, commercial .15, originality .10). Then each survivor is assigned **one** role, so
nothing competes for the same job — that is what keeps the output a synthesis instead of a
collage.

In [ ]:
from archivist.roles import assign_roles
from archivist.scoring import select_references
from archivist.trends import keywords

selected, notes = select_references(
    candidates, topic_terms=keywords(TOPIC),
    keep=settings.keep_references, minimum=settings.min_reference_score,
)
references = assign_roles(selected)

for reference in references:
    role = reference.role.value if reference.role else "reserve"
    print(f"{role:<12} {reference.score:>5.2f}  {reference.source:<10} {describe(reference.attributes)}")
for note in notes:
    print(" !", note)

In [ ]:
# The board as the app shows it.
from IPython.display import HTML, display
from archivist.board import write_board

paths = write_board(references, run_dir, title=f"{TOPIC} — reference board")
display(HTML(Path(paths["html"]).read_text(encoding="utf-8")))

## 6 · Visual DNA, art direction, style lock

The DNA is written from the measurements above. The art direction names the *system* (never
the subject) and is written to `runs/<collection>/style_lock.json` — every later run in the
collection inherits it, which is how a body of work stays recognisable while the subjects change.

In [ ]:
from archivist import direction as direction_mod
from archivist.dna import build_dna

dna = build_dna(references, ladder, garment=GARMENT, aggressiveness=AGGRESSIVENESS)
lock = direction_mod.load_lock(settings.runs_dir, settings.collection)
art_direction = direction_mod.synthesise(
    ladder, dna, references, seed=settings.seed, aggressiveness=AGGRESSIVENESS, lock=lock,
)
direction_mod.save_lock(settings.runs_dir, settings.collection, art_direction)

print(art_direction.style_name, "—", art_direction.institution)
print(art_direction.thesis, "\n")
for line in dna.as_lines():
    print(" ·", line)
print("\nstyle lock:", "inherited" if lock else "written for the first time")

## 7 · Three directions, prompts, quality gate

A safe commercial, B niche cultural, C extreme experimental — ranked on six weighted criteria.
The gate then scores eight categories; anything under 7 forces a revision that must change at
least two axes, and the change is recorded on the concept.

In [ ]:
from archivist import gate, prompts
from archivist.variations import build_concepts, recommend

concepts = build_concepts(ladder, art_direction, references,
                          aggressiveness=AGGRESSIVENESS, seed=settings.seed)
for concept in concepts:
    gate.enforce(concept, art_direction, ladder, references, seed=settings.seed, garment=GARMENT)

recommended = recommend(concepts)
for concept in concepts:
    mark = "★" if concept.key == recommended else " "
    verdict = "pass" if concept.gate["passed"] else "revised: " + ",".join(concept.gate["failing"])
    print(f"{mark} {concept.key}  {concept.lane:<22} {concept.overall:>5}  {verdict}")
    print(f"    {concept.name} — {concept.thesis[:96]}")
    for mutation in concept.mutations:
        print(f"    ↳ mutated {mutation[:88]}")

In [ ]:
chosen = next(c for c in concepts if c.key == recommended)
print(chosen.prompt)

## 8 · Generation (BFL Context)

The reference images are attached as context, each with a stated job in the prompt, so the
model borrows a property rather than a picture. Needs `BFL_API_KEY`; skipped otherwise.

In [ ]:
from archivist.bfl import BFLClient, BFLError
from archivist.pipeline import _context_paths

artwork_path = None
if settings.can_generate:
    client = BFLClient(settings.bfl_api_key, base_url=settings.bfl_base_url, model=settings.bfl_model)
    context = _context_paths(references, ["HERO", "TEXTURE", "COMPOSITION", "TYPOGRAPHY"])
    print("context images:", [Path(p).name for p in context])
    artwork_path = run_dir / "artwork" / f"{chosen.key}.png"
    artwork_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        client.generate(
            chosen.prompt, artwork_path, context_paths=context,
            aspect_ratio=settings.aspect_ratio, safety_tolerance=settings.safety_tolerance,
            on_tick=lambda fraction, message: print(f"  {message}", end="\r"),
        )
        chosen.artwork_path = str(artwork_path)
        print("\nsaved", artwork_path)
    except BFLError as exc:
        print("generation failed:", exc)
        artwork_path = None
else:
    print("no BFL_API_KEY (or offline) — skipping generation.")
    print("Everything downstream still works: the next cell falls back to a reference image")
    print("so you can see the print package behave.")

## 9 · Print package

Transparent 300 dpi print file, separation preview, ink report, the three-metre legibility
check, and a garment mockup. This is the apparel-first part: if it fails the legibility check
it does not matter how good it looks at 100%.

In [ ]:
from archivist.apparel import prepare

source_image = artwork_path or Path(references[0].local_path)   # fall back so the cell always runs
assets = prepare(source_image, run_dir / "print", key=chosen.key,
                 garment=GARMENT, width_in=settings.print_width_in, dpi=settings.print_dpi)

print(assets["print_size_in"], "·", assets["print_size_px"], "px")
for key, value in assets["report"].items():
    print(f"  {key:<20} {value}")

from IPython.display import Image as IPyImage
for label in ("legibility", "mockup"):
    if label in assets:
        print(label)
        display(IPyImage(filename=assets[label], width=380))

## 10 · Everything at once

The stages above are exactly what `pipeline.run` does; normally you call this and read the
report.

In [ ]:
# The whole bot in one call: discover, then design, with nothing typed.
from archivist.pipeline import PipelineOptions, autopilot

auto = autopilot(
    Settings.from_env(ROOT, collection=COLLECTION, offline=OFFLINE),
    PipelineOptions(garment=GARMENT, aggressiveness=AGGRESSIVENESS, generate=False),
    designs=1,
    log=print,
)
print("\ndesigned:", auto.topics)
for warning in auto.warnings:
    print(" !", warning)

In [ ]:
from archivist.pipeline import run

result = run(
    TOPIC,
    Settings.from_env(ROOT, collection=COLLECTION, offline=OFFLINE),
    PipelineOptions(
        audience=AUDIENCE, garment=GARMENT, aggressiveness=AGGRESSIVENESS,
        variants=["A", "B", "C"],
        generate=False,          # flip to True once a BFL key is set
    ),
    log=lambda line: print(line),
)
print("\nreport:", Path(result.run_dir) / "report.md")

In [ ]:
from IPython.display import Markdown

display(Markdown((Path(result.run_dir) / "report.md").read_text(encoding="utf-8")))

## 11 · Scheduling it

One theme becomes a sequence of distinct but related topics on a schedule. Every run in the
collection inherits the same style lock, so a drop reads as one body of work.

In [ ]:
from archivist.scheduler import Scheduler, plan_preview, plan_topics

topics = plan_topics("north sea oil decommissioning", 6, seed=0)
for index, when, topic in plan_preview(topics, cadence="daily", at_time="09:00"):
    print(f"{index:>2}  {when}  {topic}")

In [ ]:
# An autopilot job rediscovers topics at every firing — nothing is typed, ever.
scheduler = Scheduler(settings, on_event=print)
job = scheduler.create_job(
    "nightly autopilot", [], mode="autopilot", cadence="daily", at_time="03:00",
    options={"collection": COLLECTION, "garment": GARMENT, "generate": False,
             "offline": OFFLINE, "designs": 1},
)
print(job.id, job.describe(), "→", job.next_run)
scheduler.job_rows()

## 12 · Launch the control room

Setup, connection tests, the studio, monitoring, the scheduler and deployment — the same
engine behind a UI. `inline=True` embeds it below; use `share=True` for a temporary public
link, or run `python -m archivist.app` in a terminal.

In [ ]:
from archivist.app import build_app, styling_kwargs, _gradio_major

demo = build_app()
launch_kwargs = {"inline": True, "quiet": True}
if _gradio_major() >= 6:          # Gradio 6 moved theme/css to launch()
    launch_kwargs.update(styling_kwargs())
demo.queue().launch(**launch_kwargs)

---

**Reminder.** References are ingredients, never targets: nothing mined is reproduced in the
output. Check licensing before commercial use, keep the Pexels attribution the board records,
and keep recognisable brands, logos and real people out of what you print.